In [ ]:
import os, sys, re, json, subprocess, textwrap

def _pip(args):
    return subprocess.run(f"{sys.executable} -m pip install -q {args}",
                          shell=True).returncode == 0

try:
    import llama_cpp
except ImportError:
    print("Installing llama-cpp-python (CUDA). This can take a few minutes...")
    wheel_ok = _pip("llama-cpp-python "
                    "--extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124")
    if not wheel_ok:
        print("Prebuilt wheel unavailable — compiling from source...")
        subprocess.run(
            f'CMAKE_ARGS="-DGGML_CUDA=on" {sys.executable} -m pip install '
            f'-q llama-cpp-python --no-cache-dir --force-reinstall',
            shell=True, check=True,
        )
_pip("huggingface_hub")

from huggingface_hub import hf_hub_download
from llama_cpp import Llama

In [ ]:
REPO_ID  = "empero-ai/Qwythos-9B-Claude-Mythos-5-1M-GGUF"
FILENAME = "Qwythos-9B-Claude-Mythos-5-1M-Q4_K_M.gguf"

print(f"Downloading {FILENAME} ...")
model_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)
print("Saved to:", model_path)

llm = Llama(
    model_path=model_path,
    n_gpu_layers=-1,
    n_ctx=16384,
    n_batch=512,
    chat_format="chatml",
    verbose=False,
)

GEN = dict(temperature=0.6, top_p=0.95, top_k=20, repeat_penalty=1.05)

def split_reasoning(text: str):
    m = re.search(r"<think>(.*?)</think>(.*)", text, re.DOTALL)
    if m:
        return m.group(1).strip(), m.group(2).strip()
    return None, text.strip()

In [ ]:
print("\n" + "="*70 + "\n[5] Basic chat\n" + "="*70)
out = llm.create_chat_completion(
    messages=[
        {"role": "system", "content": "You are a concise, helpful assistant."},
        {"role": "user",   "content": "In one paragraph, explain what a GGUF file is."},
    ],
    max_tokens=1024, **GEN,
)
reasoning, answer = split_reasoning(out["choices"][0]["message"]["content"])
if reasoning:
    print("\n--- reasoning (hidden from end users) ---")
    print(textwrap.fill(reasoning, 88))
print("\n--- answer ---")
print(textwrap.fill(answer, 88))

print("\n" + "="*70 + "\n[6] Streaming\n" + "="*70)
stream = llm.create_chat_completion(
    messages=[{"role": "user",
               "content": "A farmer has 17 sheep; all but 9 run away. How many are left? Think, then answer."}],
    max_tokens=1024, stream=True, **GEN,
)
for chunk in stream:
    piece = chunk["choices"][0]["delta"].get("content", "")
    print(piece, end="", flush=True)
print()

In [ ]:
print("\n" + "="*70 + "\n[7] Multi-turn chat\n" + "="*70)
class Chat:
    def __init__(self, system="You are a helpful assistant."):
        self.messages = [{"role": "system", "content": system}]
    def say(self, user_msg, max_tokens=1024):
        self.messages.append({"role": "user", "content": user_msg})
        r = llm.create_chat_completion(messages=self.messages,
                                       max_tokens=max_tokens, **GEN)
        full = r["choices"][0]["message"]["content"]
        self.messages.append({"role": "assistant", "content": full})
        return split_reasoning(full)[1]

chat = Chat()
print("User: My favorite number is 7.")
print("Bot :", chat.say("My favorite number is 7. Remember it."))
print("\nUser: What's my favorite number squared?")
print("Bot :", chat.say("What is my favorite number squared?"))

In [ ]:
print("\n" + "="*70 + "\n[8] Tool calling\n" + "="*70)

def calculate(expression: str):
    """Evaluate a basic arithmetic expression safely."""
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:
        return {"error": "unsupported characters"}
    try:
        return {"result": eval(expression, {"__builtins__": {}}, {})}
    except Exception as e:
        return {"error": str(e)}

TOOLS = {"calculate": calculate}

TOOL_SYSTEM = (
    "You have access to tools. When a tool is needed, emit exactly:\n"
    "<tool_call><function=NAME><parameter=arg>VALUE</parameter></function></tool_call>\n"
    "Available tools:\n"
    "- calculate(expression): evaluate an arithmetic expression.\n"
    "After you receive the tool result, give the final answer to the user."
)

def chatml(messages):
    s = ""
    for m in messages:
        s += f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n"
    return s + "<|im_start|>assistant\n"

def parse_tool_call(text):
    m = re.search(r"<function=(\w+)>(.*?)</function>", text, re.DOTALL)
    if not m:
        return None
    name = m.group(1)
    args = dict(re.findall(r"<parameter=(\w+)>(.*?)</parameter>", m.group(2), re.DOTALL))
    return name, {k: v.strip() for k, v in args.items()}

msgs = [
    {"role": "system", "content": TOOL_SYSTEM},
    {"role": "user",   "content": "What is 1234 * 5678 + 42? Use the calculator."},
]
r1 = llm(chatml(msgs), max_tokens=1024,
         stop=["<|im_end|>"], **GEN)["choices"][0]["text"]
call = parse_tool_call(r1)

if call:
    name, args = call
    print(f"Model requested tool: {name}({args})")
    result = TOOLS[name](**args)
    print("Tool result:", result)
    msgs.append({"role": "assistant", "content": r1})
    msgs.append({"role": "tool", "content": json.dumps(result)})
    r2 = llm(chatml(msgs), max_tokens=1024,
             stop=["<|im_end|>"], **GEN)["choices"][0]["text"]
    print("\nFinal answer:", split_reasoning(r2)[1])
else:
    print("Model answered without a tool call:\n", split_reasoning(r1)[1])

print("\nDone. Edit prompts / swap the quant (FILENAME) / raise n_ctx to explore further.")

Installing llama-cpp-python (CUDA). This can take a few minutes...


Qwythos-9B-Claude-Mythos-5-1M-Q4_K_M.ggu(…): reconstructing file:   0%|          |  0.00B / 5.63GB            

Qwythos-9B-Claude-Mythos-5-1M-Q4_K_M.ggu(…): downloading bytes:           |  0.00B            

Saved to: /root/.cache/huggingface/hub/models--empero-ai--Qwythos-9B-Claude-Mythos-5-1M-GGUF/snapshots/5cebd89dd078b033c26988248740ddadf40f2fbd/Qwythos-9B-Claude-Mythos-5-1M-Q4_K_M.gguf

[5] Basic chat

--- reasoning (hidden from end users) ---
The user wants a concise explanation of what a GGUF file is, presented in one paragraph.
I need to cover the key points: it's a file format for storing large language model
weights, developed by the GGML project (Georgi Gerganov), optimized for CPU inference
with quantization support, and commonly used with tools like llama.cpp. It should be
straightforward and informative without unnecessary technical jargon. I'll structure the
paragraph to introduce the format, mention its purpose, highlight its efficiency
features (quantization, CPU-focused), and note its compatibility with popular
frameworks. This will give a clear overview suitable for someone asking for a brief
explanation.

--- answer ---
A GGUF (Giant Language Model Fine Tuned) file is a